# v5 FINAL — Última versión

**Resultados confirmados de v4:**
- `analyzer='char'` > `char_wb`: +0.0027
- `max_features=400k` es el mejor
- Config ganadora v4: `char(1,4), 400k, min_df=1, all42 feats` → F1=0.2737 local
- Kaggle v3: 0.29

**Lo que falta probar (interrumpido en v4):**
1. `min_df=2` vs `min_df=1` con char 400k
2. Subconjunto de features numéricas (top-10 vs todas)
3. Ajuste fino de C con `dual=False` (3-5x más rápido)
4. Entrenamiento final con 100% de datos

**Optimizaciones de velocidad:** `dual=False` + `max_iter=1000` en LinearSVC.
Esto usa más CPU y converge igual de bien con TF-IDF normalizado.

In [2]:
import pandas as pd
import numpy as np
import re, os, math, warnings
from collections import Counter
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.feature_selection import f_classif
from sklearn.metrics import f1_score
from scipy.sparse import hstack, csr_matrix
import matplotlib.pyplot as plt

SEED = 42
os.makedirs('./submissions', exist_ok=True)

# Confirmado de v4
BEST_ANALYZER  = 'char'
BEST_NGRAM     = (1, 4)
BEST_MAXF      = 400000
BASELINE_LOCAL  = 0.2737   # mejor de v4
BASELINE_KAGGLE = 0.2900   # mejor Kaggle actual

print('✅ Listo')

✅ Listo


In [3]:
df_train = pd.read_csv('./Data/train.csv')
df_eval  = pd.read_csv('./Data/eval.csv')
print(f'Train: {df_train.shape} | Eval: {df_eval.shape}')

Train: (31403, 2) | Eval: (3490, 2)


In [4]:
# ── Normalización OCR (idéntica a v3/v4) ──
CHAR_MAP = [
    ('\ufb01','fi'),('\ufb02','fl'),('\ufb00','ff'),('\ufb03','ffi'),('\ufb04','ffl'),
    ('\xe6','ae'),('\u0153','oe'),
    ('-\n',''),('- \n',''),('\xad',''),
    ('\xbb',' '),('\xab',' '),
    ('\u2018',"'"),("\u2019","'"),("\u201c",'"'),("\u201d",'"'),
    ('\xa3',' '),('\xa7',' '),('\xb6',' '),
    ('\u2020',' '),('\u2021',' '),('\u2022',' '),
    ('\u2014',' '),('\u2013',' '),
]

def normalize_ocr(text):
    text = str(text)
    for src, tgt in CHAR_MAP:
        text = text.replace(src, tgt)
    text = text.replace('\n',' ').replace('\r',' ').replace('\t',' ')
    return re.sub(r'  +', ' ', text).strip()

data = df_train.drop_duplicates(subset='text').reset_index(drop=True).copy()
data['text_clean']    = data['text'].apply(normalize_ocr)
df_eval['text_clean'] = df_eval['text'].apply(normalize_ocr)
print(f'Train sin duplicados: {len(data)}')

Train sin duplicados: 31352


In [5]:
# ── Features históricas (idénticas a v3/v4, ya validadas) ──
RE_LONG_S    = re.compile(r'[bcdfghjklmnpqrstvwxyz]f[aeiouáéíóú]', re.I)
RE_V_AS_U    = re.compile(r'\bvn[aeiouáéíóú]|\bvn\b|\bvm\b', re.I)
RE_ROMAN     = re.compile(
    r'\b(M{1,4}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})'
    r'|CM|CD|XC|XL|IX|IV|D?C{2,3}|L?X{2,3}|V?I{2,4})\b')
RE_LATIN_END = re.compile(r'\b\w{3,}(orum|ibus|atis|endi|antis|entis)\b', re.I)
RE_LATIN_NOM = re.compile(r'\b\w{3,}(um|us|ae)\b', re.I)
RE_QU_ARC    = re.compile(r'\bqu[aou]\w', re.I)
RE_SS        = re.compile(r'ss', re.I)
RE_FF        = re.compile(r'ff', re.I)
RE_CION      = re.compile(r'\b\w{3,}cion\b', re.I)
RE_TION      = re.compile(r'\b\w{3,}tion\b', re.I)
RE_SION      = re.compile(r'\b\w{3,}sion\b', re.I)
RE_ABBREV    = re.compile(r'\b[A-Za-z]{1,4}\.')
RE_MCASE     = re.compile(r'\b[A-Z][a-z]{1,}[A-Z]\w*\b')
RE_RDIAC     = re.compile(r'[àâãäāăąèêëēĕěîïīĭôõōŏùûüūŭ]', re.I)
RE_SEMI      = re.compile(r';')
RE_COLON     = re.compile(r':')
RE_PAREN     = re.compile(r'[()]')

def extract_features(text):
    words   = text.split()
    n       = max(len(words), 1)
    nc      = max(len(text), 1)
    lengths = [len(w) for w in words]
    counts  = Counter(text)
    total   = len(text) or 1
    entropy = -sum((c/total)*math.log2(c/total) for c in counts.values()) if text else 0
    vowels  = sum(1 for c in text.lower() if c in 'aeiouáéíóúàèìòùäëïöüâêîôû')
    cons    = sum(1 for c in text.lower() if c.isalpha()
                  and c not in 'aeiouáéíóúàèìòùäëïöüâêîôû')
    sents   = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    ns      = max(len(sents), 1)
    return {
        'wl_mean':    np.mean(lengths) if lengths else 0,
        'wl_std':     np.std(lengths)  if lengths else 0,
        'wl_p75':     np.percentile(lengths, 75) if lengths else 0,
        'wl_p90':     np.percentile(lengths, 90) if lengths else 0,
        'ratio_long': sum(1 for l in lengths if l > 10) / n,
        'ratio_short':sum(1 for l in lengths if l <= 2) / n,
        'ratio_med':  sum(1 for l in lengths if 3 <= l <= 6) / n,
        'char_entropy':   entropy,
        'cv_ratio':       cons / vowels if vowels > 0 else 0,
        'word_char_ratio':n / nc,
        'comma_rate':     text.count(',') / n,
        'period_rate':    text.count('.') / n,
        'semicolon_rate': len(RE_SEMI.findall(text))  / n,
        'colon_rate':     len(RE_COLON.findall(text)) / n,
        'paren_rate':     len(RE_PAREN.findall(text)) / n,
        'excl_rate':      text.count('!') / n,
        'quest_rate':     text.count('?') / n,
        'total_punct':    sum(1 for c in text if c in '.,;:!?()[]{}') / n,
        'long_s_rate':    len(RE_LONG_S.findall(text))    / n,
        'v_as_u_rate':    len(RE_V_AS_U.findall(text))    / n,
        'latin_case':     len(RE_LATIN_END.findall(text)) / n,
        'latin_nom':      len(RE_LATIN_NOM.findall(text)) / n,
        'qu_archaic':     len(RE_QU_ARC.findall(text))    / n,
        'ss_rate':        len(RE_SS.findall(text))         / n,
        'ff_rate':        len(RE_FF.findall(text))         / n,
        'cion_rate':      len(RE_CION.findall(text))       / n,
        'tion_rate':      len(RE_TION.findall(text))       / n,
        'sion_rate':      len(RE_SION.findall(text))       / n,
        'abbrev_rate':    len(RE_ABBREV.findall(text))     / n,
        'roman_rate':     len(RE_ROMAN.findall(text))      / n,
        'rare_diac':      len(RE_RDIAC.findall(text))      / nc,
        'mixed_case':     len(RE_MCASE.findall(text))      / n,
        'ttr':         len(set(w.lower() for w in words)) / n,
        'upper_ratio': sum(1 for c in text if c.isupper()) / nc,
        'digit_ratio': sum(1 for c in text if c.isdigit()) / nc,
        'alpha_ratio': sum(1 for c in text if c.isalpha()) / nc,
        'all_caps':    sum(1 for w in words if w.isupper() and len(w)>1) / n,
        'pure_digit':  sum(1 for w in words if w.isdigit()) / n,
        'n_words':     float(n),
        'avg_sent_len':n / ns,
        'n_sentences': float(ns),
    }

print('Extrayendo features train...')
feats_train = pd.DataFrame(
    data['text_clean'].apply(extract_features).tolist()
).fillna(0)
print('Extrayendo features eval...')
feats_eval = pd.DataFrame(
    df_eval['text_clean'].apply(extract_features).tolist()
).fillna(0)

ALL_FEATS = feats_train.columns.tolist()
print(f'✅ Features: {len(ALL_FEATS)}')

Extrayendo features train...
Extrayendo features eval...
✅ Features: 41


In [6]:
# ── Top features por F-score ──
f_scores, _ = f_classif(feats_train.values, data['decade'].values)
df_disc = pd.DataFrame({'feature': ALL_FEATS, 'f_score': f_scores})\
            .sort_values('f_score', ascending=False).reset_index(drop=True)
TOP_10 = df_disc.head(10)['feature'].tolist()
print('Top 10 features:', TOP_10)

Top 10 features: ['wl_std', 'wl_mean', 'ratio_long', 'word_char_ratio', 'wl_p90', 'comma_rate', 'char_entropy', 'long_s_rate', 'colon_rate', 'ratio_short']


In [7]:
# ── Partición ──
X_all = pd.concat([
    data['text_clean'].reset_index(drop=True),
    feats_train.reset_index(drop=True)
], axis=1)
y_all = data['decade'].reset_index(drop=True)

X_eval_full = pd.concat([
    df_eval['text_clean'].reset_index(drop=True),
    feats_eval.reset_index(drop=True)
], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

Train: (25081, 42) | Test: (6271, 42)


In [8]:
# ── Función de construcción de matrices ──
# dual=False es 3-5x más rápido cuando n_features >> n_samples
# max_iter=1000 suficiente para TF-IDF sublinear normalizado

def make_vec(min_df=1, max_f=400000):
    return TfidfVectorizer(
        analyzer=BEST_ANALYZER,
        ngram_range=BEST_NGRAM,
        sublinear_tf=True,
        max_features=max_f,
        min_df=min_df,
        max_df=0.98,
        strip_accents=None,
    )

def build_matrices(vec, feat_cols, fit_vec=True,
                   X_tr=None, X_te=None, X_ev=None):
    """Construye matrices train/test/eval combinando TF-IDF + features numéricas."""
    X_tr = X_tr if X_tr is not None else X_train
    X_te = X_te if X_te is not None else X_test
    X_ev = X_ev if X_ev is not None else X_eval_full

    if fit_vec:
        tfidf_tr = vec.fit_transform(X_tr['text_clean'])
    else:
        tfidf_tr = vec.transform(X_tr['text_clean'])
    tfidf_te = vec.transform(X_te['text_clean'])
    tfidf_ev = vec.transform(X_ev['text_clean'])

    if feat_cols:
        sc = StandardScaler()
        num_tr = csr_matrix(sc.fit_transform(X_tr[feat_cols].values))
        num_te = csr_matrix(sc.transform(X_te[feat_cols].values))
        num_ev = csr_matrix(sc.transform(X_ev[feat_cols].values))
        return (hstack([tfidf_tr, num_tr]),
                hstack([tfidf_te, num_te]),
                hstack([tfidf_ev, num_ev]))
    return tfidf_tr, tfidf_te, tfidf_ev

def fit_eval(name, mat_tr, mat_te, C=0.5, cw=None):
    clf = LinearSVC(C=C, max_iter=1000, random_state=SEED,
                    dual=False, class_weight=cw)
    clf.fit(mat_tr, y_train)
    f1 = f1_score(y_test, clf.predict(mat_te), average='macro')
    delta = f1 - BASELINE_LOCAL
    mark = '✅' if delta > 0 else '⚠️'
    print(f'{mark} {name:<50} F1={f1:.4f}  Δ={delta:+.4f}')
    return f1, clf

print(f'Baseline confirmado v4: F1={BASELINE_LOCAL:.4f}')
print(f'Config base: analyzer=char, (1,4), 400k, min_df=1, all{len(ALL_FEATS)} feats, C=0.5')
print('-' * 70)

Baseline confirmado v4: F1=0.2737
Config base: analyzer=char, (1,4), 400k, min_df=1, all41 feats, C=0.5
----------------------------------------------------------------------


In [ ]:
# ══════════════════════════════════════════════════════════
# EJE 3: min_df — ¿filtrar n-gramas únicos ayuda?
# ══════════════════════════════════════════════════════════
# min_df=1: incluye n-gramas que aparecen en UN solo texto
#           muchos son ruido OCR específico de ese documento
# min_df=2: requiere aparecer en al menos 2 documentos
#           filtra el ruido sin perder señal real
print('=== EJE 3: min_df ===')

vec_mdf1 = make_vec(min_df=1, max_f=400000)
tr1, te1, ev1 = build_matrices(vec_mdf1, ALL_FEATS)
f1_mdf1, _ = fit_eval('char(1,4) 400k min_df=1 all_feats [baseline]', tr1, te1)

vec_mdf2 = make_vec(min_df=2, max_f=400000)
tr2, te2, ev2 = build_matrices(vec_mdf2, ALL_FEATS)
f1_mdf2, _ = fit_eval('char(1,4) 400k min_df=2 all_feats',             tr2, te2)

# Ganador de EJE 3
if f1_mdf2 > f1_mdf1:
    BEST_MINDF = 2
    mat_tr_e3, mat_te_e3, mat_ev_e3 = tr2, te2, ev2
    print(f'→ min_df=2 gana')
else:
    BEST_MINDF = 1
    mat_tr_e3, mat_te_e3, mat_ev_e3 = tr1, te1, ev1
    print(f'→ min_df=1 gana')

=== EJE 3: min_df ===


In [ ]:
# ══════════════════════════════════════════════════════════
# EJE 4: features numéricas
# ══════════════════════════════════════════════════════════
# Probamos 3 variantes sobre la mejor config de EJE 3
print('=== EJE 4: features numéricas ===')
print(f'(Usando min_df={BEST_MINDF})')

vec_e4 = make_vec(min_df=BEST_MINDF, max_f=400000)

# Sin features numéricas
tr_none, te_none, ev_none = build_matrices(vec_e4, [])
f1_none, _ = fit_eval('char(1,4) 400k — SIN features numéricas', tr_none, te_none)

# Top-10 features
tr_top10, te_top10, ev_top10 = build_matrices(vec_e4, TOP_10, fit_vec=False)
f1_top10, _ = fit_eval('char(1,4) 400k — top-10 features',       tr_top10, te_top10)

# Todas las features
tr_all, te_all, ev_all = build_matrices(vec_e4, ALL_FEATS, fit_vec=False)
f1_all, _ = fit_eval(f'char(1,4) 400k — todas {len(ALL_FEATS)} features', tr_all, te_all)

# Ganador EJE 4
eje4_scores = {
    'none':  (f1_none,  tr_none,  te_none,  ev_none,  []),
    'top10': (f1_top10, tr_top10, te_top10, ev_top10, TOP_10),
    'all':   (f1_all,   tr_all,   te_all,   ev_all,   ALL_FEATS),
}
best_e4_key = max(eje4_scores, key=lambda k: eje4_scores[k][0])
BEST_F1_E4, mat_tr_best, mat_te_best, mat_ev_best, BEST_FEAT_COLS = eje4_scores[best_e4_key]
print(f'\n→ Ganador EJE 4: {best_e4_key} — F1={BEST_F1_E4:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════
# AJUSTE FINO DE C
# ══════════════════════════════════════════════════════════
# Solo 5 valores alrededor del 0.5 conocido
# dual=False → usa el solver primal, mucho más rápido con muchas features
print('=== Ajuste fino de C ===')
print(f'(Sobre la mejor config: min_df={BEST_MINDF}, feats={best_e4_key})')

best_C, best_f1_C = 0.5, 0.0
for C in [0.2, 0.3, 0.5, 0.7, 1.0]:
    clf = LinearSVC(C=C, max_iter=1000, random_state=SEED, dual=False)
    clf.fit(mat_tr_best, y_train)
    f1 = f1_score(y_test, clf.predict(mat_te_best), average='macro')
    mark = '  ← MEJOR' if f1 > best_f1_C else ''
    print(f'  C={C}  F1={f1:.4f}{mark}')
    if f1 > best_f1_C:
        best_f1_C, best_C = f1, C

print(f'\nMejor C={best_C} → F1={best_f1_C:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════
# class_weight='balanced'
# ══════════════════════════════════════════════════════════
# Con 39 clases casi uniformes no debería ayudar,
# pero lo verificamos en un fit
clf_bal = LinearSVC(C=best_C, max_iter=1000, random_state=SEED,
                    dual=False, class_weight='balanced')
clf_bal.fit(mat_tr_best, y_train)
f1_bal = f1_score(y_test, clf_bal.predict(mat_te_best), average='macro')

print(f'class_weight=balanced: F1={f1_bal:.4f}')
print(f'class_weight=None:     F1={best_f1_C:.4f}')

FINAL_CW = 'balanced' if f1_bal > best_f1_C else None
FINAL_F1 = max(f1_bal, best_f1_C)
print(f'→ Usar class_weight={FINAL_CW}  (F1 final local: {FINAL_F1:.4f})')

In [ ]:
# ══════════════════════════════════════════════════════════
# RESUMEN DE EXPERIMENTOS
# ══════════════════════════════════════════════════════════
print('=' * 60)
print('RESUMEN DE EXPERIMENTOS v5')
print('=' * 60)
print(f'Baseline v4 (local):   {BASELINE_LOCAL:.4f}')
print(f'Kaggle actual:         {BASELINE_KAGGLE:.4f}')
print()
print(f'EJE 3 — min_df ganador:  {BEST_MINDF}')
print(f'EJE 4 — feats ganador:   {best_e4_key} ({len(BEST_FEAT_COLS)} features)')
print(f'C óptimo:                {best_C}')
print(f'class_weight:            {FINAL_CW}')
print()
print(f'F1 local final:          {FINAL_F1:.4f}')
print(f'Mejora sobre baseline:   {FINAL_F1 - BASELINE_LOCAL:+.4f}')
print(f'Proyección Kaggle:      ~{BASELINE_KAGGLE + (FINAL_F1 - BASELINE_LOCAL):.4f}')
print('=' * 60)

In [ ]:
# ══════════════════════════════════════════════════════════
# ENTRENAMIENTO FINAL — 100% DE LOS DATOS
# ══════════════════════════════════════════════════════════
print('Entrenando modelo final con 100% de los datos...')
print(f'  analyzer:    {BEST_ANALYZER}')
print(f'  ngram:       {BEST_NGRAM}')
print(f'  max_features:{BEST_MAXF}')
print(f'  min_df:      {BEST_MINDF}')
print(f'  feat_cols:   {len(BEST_FEAT_COLS)} features ({best_e4_key})')
print(f'  C:           {best_C}')
print(f'  dual:        False')
print(f'  class_weight:{FINAL_CW}')

X_full = pd.concat([
    data['text_clean'].reset_index(drop=True),
    feats_train.reset_index(drop=True)
], axis=1)
y_full = data['decade'].reset_index(drop=True)

# Vectorizador final
vec_final = make_vec(min_df=BEST_MINDF, max_f=BEST_MAXF)
tfidf_full = vec_final.fit_transform(X_full['text_clean'])
tfidf_eval = vec_final.transform(X_eval_full['text_clean'])

# Features numéricas
if BEST_FEAT_COLS:
    sc_final   = StandardScaler()
    num_full   = csr_matrix(sc_final.fit_transform(X_full[BEST_FEAT_COLS].values))
    num_eval   = csr_matrix(sc_final.transform(X_eval_full[BEST_FEAT_COLS].values))
    X_full_mat = hstack([tfidf_full, num_full])
    X_eval_mat = hstack([tfidf_eval, num_eval])
else:
    X_full_mat = tfidf_full
    X_eval_mat = tfidf_eval

# Modelo final
clf_final = LinearSVC(C=best_C, max_iter=1000, random_state=SEED,
                      dual=False, class_weight=FINAL_CW)
clf_final.fit(X_full_mat, y_full)
print('✅ Modelo final entrenado')

In [ ]:
# ── Submission ──
preds = clf_final.predict(X_eval_mat)

sub = pd.DataFrame({'id': df_eval['id'], 'answer': preds})
sub.to_csv('./submissions/SUBMISSION_v5_FINAL.csv', index=False)

print('✅ SUBMISSION_v5_FINAL.csv generado')
print(f'Predicciones: {len(preds)} | Clases únicas: {len(set(preds))}')
print()
print('═' * 60)
print('RESUMEN FINAL')
print('═' * 60)
print(f'Score Kaggle anterior:   {BASELINE_KAGGLE}')
print(f'F1 local v5:             {FINAL_F1:.4f}')
print(f'Mejora local:            {FINAL_F1 - BASELINE_LOCAL:+.4f}')
print(f'Proyección Kaggle:      ~{BASELINE_KAGGLE + (FINAL_F1 - BASELINE_LOCAL):.4f}')
print('═' * 60)
print()
print('→ SUBIR: SUBMISSION_v5_FINAL.csv')

In [10]:
# ── Variables ganadoras de la exploración (v4/v5) ──
BEST_MINDF     = 1
BEST_FEAT_COLS = ALL_FEATS
best_C         = 0.5
FINAL_CW       = None
best_e4_key    = 'all'

print('✅ Variables ganadoras definidas')
print(f'  min_df:      {BEST_MINDF}')
print(f'  feat_cols:   {len(BEST_FEAT_COLS)} features')
print(f'  C:           {best_C}')
print(f'  class_weight:{FINAL_CW}')

✅ Variables ganadoras definidas
  min_df:      1
  feat_cols:   41 features
  C:           0.5
  class_weight:None


In [11]:
# ══════════════════════════════════════════════════════════
# ENTRENAMIENTO FINAL — 100% DE LOS DATOS
# char TF-IDF (suyo) + word TF-IDF (nuestro) + features numéricas
# ══════════════════════════════════════════════════════════
print('Entrenando modelo final con 100% de los datos...')
print(f'  analyzer:    {BEST_ANALYZER}')
print(f'  ngram:       {BEST_NGRAM}')
print(f'  max_features:{BEST_MAXF}')
print(f'  min_df:      {BEST_MINDF}')
print(f'  feat_cols:   {len(BEST_FEAT_COLS)} features ({best_e4_key})')
print(f'  C:           {best_C}')
print(f'  dual:        False')
print(f'  class_weight:{FINAL_CW}')
print(f'  word TF-IDF: ngram=(1,2), max_features=100k  ← AGREGADO')

X_full = pd.concat([
    data['text_clean'].reset_index(drop=True),
    feats_train.reset_index(drop=True)
], axis=1)
y_full = data['decade'].reset_index(drop=True)

# ── Char TF-IDF (suyo, intacto) ──
vec_final  = make_vec(min_df=BEST_MINDF, max_f=BEST_MAXF)
tfidf_full = vec_final.fit_transform(X_full['text_clean'])
tfidf_eval = vec_final.transform(X_eval_full['text_clean'])

# ── Word TF-IDF (nuestro, agregado) ──
word_vec   = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=100_000,
    sublinear_tf=True,
)
word_full  = word_vec.fit_transform(X_full['text_clean'])
word_eval  = word_vec.transform(X_eval_full['text_clean'])

print(f'\nShapes:')
print(f'  char TF-IDF:  {tfidf_full.shape}')
print(f'  word TF-IDF:  {word_full.shape}')

# ── Features numéricas (suyas, intactas) ──
if BEST_FEAT_COLS:
    sc_final   = StandardScaler()
    num_full   = csr_matrix(sc_final.fit_transform(X_full[BEST_FEAT_COLS].values))
    num_eval   = csr_matrix(sc_final.transform(X_eval_full[BEST_FEAT_COLS].values))
    X_full_mat = hstack([tfidf_full, word_full, num_full])
    X_eval_mat = hstack([tfidf_eval, word_eval, num_eval])
else:
    X_full_mat = hstack([tfidf_full, word_full])
    X_eval_mat = hstack([tfidf_eval, word_eval])

print(f'  combinado:    {X_full_mat.shape}')

# ── Modelo final (suyo, intacto) ──
clf_final = LinearSVC(C=best_C, max_iter=1000, random_state=SEED,
                      dual=False, class_weight=FINAL_CW)
clf_final.fit(X_full_mat, y_full)
print('✅ Modelo final entrenado')

Entrenando modelo final con 100% de los datos...
  analyzer:    char
  ngram:       (1, 4)
  max_features:400000
  min_df:      1
  feat_cols:   41 features (all)
  C:           0.5
  dual:        False
  class_weight:None
  word TF-IDF: ngram=(1,2), max_features=100k  ← AGREGADO

Shapes:
  char TF-IDF:  (31352, 400000)
  word TF-IDF:  (31352, 100000)
  combinado:    (31352, 500041)
✅ Modelo final entrenado


In [12]:
# ── Submission ──
preds = clf_final.predict(X_eval_mat)

sub = pd.DataFrame({'id': df_eval['id'], 'answer': preds})
sub.to_csv('./submissions/SUBMISSION_v5_word_char.csv', index=False)

print('✅ SUBMISSION_v5_word_char.csv generado')
print(f'Predicciones: {len(preds)} | Clases únicas: {len(set(preds))}')

✅ SUBMISSION_v5_word_char.csv generado
Predicciones: 3490 | Clases únicas: 39
